In [0]:
df = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load("/Volumes/dbacademy/default/week7_data/Superstore.csv")

display(df)

In [0]:
df = df.toDF(*[
    col.replace(" ", "_")
    for col in df.columns
])

display(df)

In [0]:
df.write \
    .format("delta") \
    .mode("overwrite") \
    .save("/Volumes/dbacademy/default/week7_data/superstore_delta")

In [0]:
delta_df = spark.read.format("delta") \
    .load("/Volumes/dbacademy/default/week7_data/superstore_delta")

display(delta_df)

In [0]:
print("Total Records :", delta_df.count())

In [0]:
from pyspark.sql.functions import col, sum, when

null_df = delta_df.select([
    sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in delta_df.columns
])

display(null_df)

In [0]:
clean_df = delta_df.dropDuplicates()

In [0]:
print("Before Cleaning :", delta_df.count())
print("After Removing Duplicates :", clean_df.count())

In [0]:
clean_df.write \
    .format("delta") \
    .mode("overwrite") \
    .save("/Volumes/dbacademy/default/week7_data/clean_superstore_delta")

In [0]:
existing_data = clean_df.limit(10)
display(existing_data)

In [0]:
from pyspark.sql.functions import col

updated_existing = existing_data \
    .withColumn("Profit", col("Profit") + 1000)

display(updated_existing)

In [0]:
from pyspark.sql.functions import col

new_records = clean_df.limit(5) \
    .withColumn("Row_ID", col("Row_ID") + 1000)

display(new_records)

In [0]:
incremental_df = updated_existing.unionByName(new_records)

display(incremental_df)

In [0]:
print("Incremental Records:", incremental_df.count())

In [0]:
incremental_df.write \
    .format("delta") \
    .mode("overwrite") \
    .save("/Volumes/dbacademy/default/week7_data/incremental_superstore_delta")

In [0]:
from delta.tables import DeltaTable

In [0]:
delta_table = DeltaTable.forPath(
    spark,
    "/Volumes/dbacademy/default/week7_data/clean_superstore_delta"
)

In [0]:
(
    delta_table.alias("target")
    .merge(
        incremental_df.alias("source"),
        "target.Row_ID = source.Row_ID"
    )
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute()
)

In [0]:
merged_df = spark.read.format("delta").load(
    "/Volumes/dbacademy/default/week7_data/clean_superstore_delta"
)

display(merged_df)

In [0]:
print("Final Record Count :", merged_df.count())

In [0]:
display(
    merged_df.filter("Row_ID <= 10")
)

In [0]:
display(
    merged_df.filter("Row_ID >= 1000")
)

In [0]:
print("Final Record Count:", merged_df.count())

In [0]:
from pyspark.sql.functions import count

duplicates = merged_df.groupBy("Row_ID") \
    .agg(count("*").alias("count")) \
    .filter("count > 1")

display(duplicates)

In [0]:
display(merged_df)

In [0]:
customer_master = clean_df

print("Master Records:", customer_master.count())
display(customer_master)

In [0]:
customer_master.coalesce(1) \
    .write \
    .mode("overwrite") \
    .option("header", "true") \
    .csv("/Volumes/dbacademy/default/week7_data/customer_master_single")

In [0]:
customer_incremental = incremental_df

print("Incremental Records:", customer_incremental.count())
display(customer_incremental)

In [0]:
customer_incremental.coalesce(1) \
    .write \
    .mode("overwrite") \
    .option("header", "true") \
    .csv("/Volumes/dbacademy/default/week7_data/customer_incremental_single")